# UHI-E3LLM raw-data preprocessing

This notebook rebuilds the model-ready dataset from the public raw sources cited in Section 4.2 of the article. Only two inputs are downloaded locally before starting:

1. **Monthly UHI GeoTIFFs (AMod2, daytime)**: [Figshare dataset](https://figshare.com/articles/dataset/Global_Urban_Heat_Island_Intensity_Dataset/24821538). Arrange them as `AMod2/AMod2_Day_<YEAR>_<MONTH>.tif` for 2003-2020.
2. **Administrative city boundaries**: [iEarth DataHub](https://data-starcloud.pcl.ac.cn/iearthdata/14). Use the article's city-boundary product containing the `urbanArea` field and retain its original feature order.

AgERA5, MOD13A3, PM2.5, and GHSL are read directly through Google Earth Engine (GEE), using the catalog links shown below. Register a GEE project before running those cells. The complete processed release remains available at [Zenodo](https://doi.org/10.5281/zenodo.22161925).

Run the cells in order. Long extraction cells are resumable: an existing CSV is skipped only after its complete ordered city-ID column has been verified.


## 1. Configuration and source inventory


In [ ]:
from pathlib import Path
import logging
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts" / "preprocess_raw_data.py").is_file():
    if (REPO_ROOT / "preprocess_raw_data.py").is_file():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError("Start Jupyter from the repository root or scripts/ directory")

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from preprocess_raw_data import (
    AGERA5_ASSET, CLIMATE_BANDS, GHS_EPOCHS, MOD13A3_ASSET, PM25_ASSET,
    describe_sources, extract_gee_epoch_images, extract_gee_land_use,
    extract_gee_monthly_collection, extract_uhi_rasters,
    initialize_earth_engine, make_ee_features, prepare_city_boundaries,
    reshape_uhi_by_city, validate_raw_layout,
)

# Change only this block.
DOWNLOAD_ROOT = Path("/absolute/path/to/downloaded_raw_sources")
URBAN_BOUNDARY_FILE = DOWNLOAD_ROOT / "urban_boundaries" / "urban_boundaries.geojson"
UHI_RASTER_ROOT = DOWNLOAD_ROOT / "uhi"
GEE_PROJECT = "your-google-earth-engine-project"
AUTHENTICATE = False       # True for the first run on a new workstation
OVERWRITE = False          # Existing complete outputs are reused
EXPECTED_CITIES = 1020
GEE_WORKERS = 4            # Reduce if the GEE account is rate-limited

DATA_DIR = REPO_ROOT / "data"
RAW_OUTPUT = DATA_DIR / "raw"
FILTERED_BOUNDARIES = RAW_OUTPUT / "boundaries" / "eligible_cities_1020.geojson"
CITY_METRICS = DATA_DIR / "city_metrics" / "city_metrics_all.csv"
UHI_MONTHLY = RAW_OUTPUT / "UHI_monthly"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
describe_sources()


### Required local layout

```text
<DOWNLOAD_ROOT>/
|-- urban_boundaries/
|   `-- urban_boundaries.geojson
`-- uhi/
    `-- AMod2/
        |-- AMod2_Day_2003_1.tif
        |-- ...
        `-- AMod2_Day_2020_12.tif
```

The boundary file must have a CRS and an `urbanArea` column in km2. Cities with an area of at least 100 km2 are retained. IDs `0..1019` are assigned once in source-feature order and then carried explicitly through every local and GEE reduction.


## 2. Filter boundaries and build city metadata


In [ ]:
import geopandas as gpd
import pandas as pd

if FILTERED_BOUNDARIES.is_file() and CITY_METRICS.is_file() and not OVERWRITE:
    cities = gpd.read_file(FILTERED_BOUNDARIES).sort_values("id").reset_index(drop=True)
    city_metrics = pd.read_csv(CITY_METRICS).sort_values("id").reset_index(drop=True)
else:
    cities, city_metrics = prepare_city_boundaries(
        URBAN_BOUNDARY_FILE,
        FILTERED_BOUNDARIES,
        CITY_METRICS,
        expected_cities=EXPECTED_CITIES,
        overwrite=OVERWRITE,
    )

assert cities["id"].astype(int).tolist() == list(range(EXPECTED_CITIES))
city_metrics.head()


Geometry validity and perimeters are evaluated geodesically on WGS84, so global cities are not forced into one local UTM zone. `Area` remains the source `urbanArea` value because it is the area definition used for the five manuscript size strata; `Geodesic Area` is retained as an audit column.


## 3. Initialize Earth Engine and explicit city features


In [ ]:
ee = initialize_earth_engine(GEE_PROJECT, authenticate=AUTHENTICATE)
gee_cities = make_ee_features(cities)
print(f"Prepared {len(gee_cities)} GEE features with explicit city_id properties")


## 4. Extract monthly UHI from the downloaded GeoTIFFs


In [ ]:
extract_uhi_rasters(
    UHI_RASTER_ROOT,
    cities,
    UHI_MONTHLY,
    dataset="AMod2",
    period="Day",
    factor=0.01,
    overwrite=OVERWRITE,
)
reshape_uhi_by_city(
    UHI_MONTHLY,
    RAW_OUTPUT / "Extracted_City_Data",
    expected_cities=EXPECTED_CITIES,
    dataset="AMod2",
    overwrite=OVERWRITE,
)


Raster polygons are reprojected to each source raster CRS before masking. Means use only valid in-polygon pixels; raster NoData and pixels outside the polygon are excluded. The Figshare integer UHI values are multiplied by `0.01` as specified by the source data.


## 5. Extract monthly dynamic factors from GEE (2003-2020)


In [ ]:
extract_gee_monthly_collection(
    AGERA5_ASSET, CLIMATE_BANDS, gee_cities,
    RAW_OUTPUT / "ClimateData", "climate",
    scale=1000, block_size=10, max_workers=GEE_WORKERS,
    overwrite=OVERWRITE,
)

extract_gee_monthly_collection(
    MOD13A3_ASSET, ("NDVI", "EVI"), gee_cities,
    RAW_OUTPUT / "MOD13A3_EVI_NDVI", "evi_ndvi",
    factor=0.0001, scale=1000, block_size=4,
    max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)

extract_gee_monthly_collection(
    PM25_ASSET, ("b1",), gee_cities,
    RAW_OUTPUT / "PM25", "pm25",
    scale=1000, block_size=4, max_workers=GEE_WORKERS,
    overwrite=OVERWRITE,
)


The AgERA5 asset and its band schema are frozen to those used for the manuscript dataset. MOD13A3's documented scale factor `0.0001` is applied to NDVI and EVI. Monthly files always contain exactly one row per city, sorted by `id`.


## 6. Extract structural factors from GEE


In [ ]:
extract_gee_epoch_images(
    "JRC/GHSL/P2023A/GHS_POP/{year}", "population_count", GHS_EPOCHS,
    gee_cities, RAW_OUTPUT / "GHS_POP", "GHS_POP",
    scale=1000, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)
extract_gee_epoch_images(
    "JRC/GHSL/P2023A/GHS_BUILT_S/{year}", "built_surface", GHS_EPOCHS,
    gee_cities, RAW_OUTPUT / "GHS_BUILT_S", "GHS_BUILT_S",
    scale=1000, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)
extract_gee_epoch_images(
    "JRC/GHSL/P2023A/GHS_BUILT_V/{year}", "built_volume_total", GHS_EPOCHS,
    gee_cities, RAW_OUTPUT / "GHS_BUILT_V", "GHS_BUILT_V",
    scale=1000, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)
extract_gee_epoch_images(
    "JRC/GHSL/P2023A/GHS_BUILT_H/{year}", "built_height", (2018,),
    gee_cities, RAW_OUTPUT / "GHS_BUILT_H", "ghs_built_h",
    scale=1000, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)
extract_gee_epoch_images(
    "JRC/GHSL/P2023A/GHS_SMOD_V2-0/{year}", "smod_code", GHS_EPOCHS,
    gee_cities, RAW_OUTPUT / "GHS_SMOD", "ghs_smod",
    scale=1000, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)
extract_gee_land_use(
    gee_cities,
    RAW_OUTPUT / "GHS_BUILT_C" / "ghs_built_c_2018.csv",
    scale=10, max_workers=GEE_WORKERS, overwrite=OVERWRITE,
)


GHS-POP, GHS-BUILT-S, GHS-BUILT-V, and GHS-SMOD are extracted for 2005, 2010, 2015, and 2020. Built height and built characteristics use 2018. The 16 GHS-BUILT-C classes are retained and additionally aggregated into the five prompt groups: Open Spaces, Water Surfaces, Road Surfaces, Residential, and Non-Residential.


## 7. Validate extraction and build Hugging Face datasets


In [ ]:
validate_raw_layout(RAW_OUTPUT, CITY_METRICS, expected_cities=EXPECTED_CITIES)

subprocess.run([
    sys.executable, str(REPO_ROOT / "scripts" / "build_hf_dataset_from_raw.py"),
    "--raw-data", str(RAW_OUTPUT),
    "--city-metrics", str(CITY_METRICS),
    "--output", str(DATA_DIR),
    "--expected-cities", str(EXPECTED_CITIES),
], check=True, cwd=REPO_ROOT)


For each target month from 2004-01 through 2020-12, the builder uses the immediately preceding 12 months for historical UHI and all monthly factors. Targets through 2017-12 produce 168 training samples per city; 2018-01 through 2020-12 produce 36 evaluation samples per city. GHS epochs use the frozen rule in `build_hf_dataset_from_raw.py`.


## 8. Generate the 500/100 split, calibration cohort, and regional folds


In [ ]:
sample_command = [
    sys.executable, str(REPO_ROOT / "scripts" / "sample_modeling_cities.py"),
    "--city-metrics", str(CITY_METRICS),
    "--output-dir", str(DATA_DIR / "city_metrics"),
]
if OVERWRITE:
    sample_command.append("--overwrite")
subprocess.run(sample_command, check=True, cwd=REPO_ROOT)

subprocess.run([
    sys.executable, str(REPO_ROOT / "scripts" / "build_city_manifests.py"),
    "--data_dir", str(DATA_DIR),
], check=True, cwd=REPO_ROOT)

subprocess.run([
    sys.executable, str(REPO_ROOT / "scripts" / "validate_preprocessed_data.py"),
    "--data-dir", str(DATA_DIR), "--deep",
], check=True, cwd=REPO_ROOT)


The default sampler draws 600 modeling cities by the five city-size strata, allocates 500 to training and 100 to held-out testing within those strata, and draws another 100 calibration cities from the remaining candidates. A fresh seed is generated and printed; pass an explicit `--seed` to reproduce a newly sampled cohort. Regional labels and all twelve leave-one-region-out manifests are then derived from coordinates.
